In [127]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from haversine import haversine, Unit
import folium
from pathlib import Path



In [128]:
# Config
DATA_PATH = './VED-master/Data/VED_DynamicData_Part1/VED_171101_week.csv'
OUTPUT_MAPS_DIR = './trip_maps/'
VEHICLE_ID = 10 #To work with an EV

# Make output directory
Path(OUTPUT_MAPS_DIR).mkdir(parents=True, exist_ok=True)

In [129]:
data = pd.read_csv(DATA_PATH)
data = data[data['VehId'] == VEHICLE_ID]
data.head()

,DayNum,VehId,Trip,Timestamp(ms),Latitude[deg],Longitude[deg],Vehicle Speed[km/h],MAF[g/sec],Engine RPM[RPM],Absolute Load[%],...,Air Conditioning Power[kW],Air Conditioning Power[Watts],Heater Power[Watts],HV Battery Current[A],HV Battery SOC[%],HV Battery Voltage[V],Short Term Fuel Trim Bank 1[%],Short Term Fuel Trim Bank 2[%],Long Term Fuel Trim Bank 1[%],Long Term Fuel Trim Bank 2[%]
416,1.719774,10,1558,0,42.277066,-83.763404,53.590000,NaN,NaN,NaN,...,NaN,0.0,2250.0,-21.5,96.341469,386.0,NaN,NaN,NaN,NaN
417,1.719774,10,1558,200,42.277066,-83.763404,51.980000,NaN,NaN,NaN,...,NaN,0.0,2250.0,-21.5,96.341469,386.0,NaN,NaN,NaN,NaN
418,1.719774,10,1558,1200,42.277066,-83.763404,50.369999,NaN,NaN,NaN,...,NaN,0.0,2250.0,-21.5,96.341469,386.0,NaN,NaN,NaN,NaN
419,1.719774,10,1558,1500,42.277066,-83.763404,50.369999,NaN,NaN,NaN,...,NaN,0.0,2250.0,23.5,96.341469,390.5,NaN,NaN,NaN,NaN
420,1.719774,10,1558,2300,42.277066,-83.763404,49.799999,NaN,NaN,NaN,...,NaN,0.0,2250.0,23.5,96.341469,390.5,NaN,NaN,NaN,NaN


In [130]:
def calculate_trip_distance(trip_df):
    """Calculate total distance (km) of a trip using GPS coordinates."""
    total_distance = 0.0
    for i in range(1, len(trip_df)):
        coord1 = (trip_df['Latitude[deg]'].iloc[i-1], trip_df['Longitude[deg]'].iloc[i-1])
        coord2 = (trip_df['Latitude[deg]'].iloc[i], trip_df['Longitude[deg]'].iloc[i])
        total_distance += haversine(coord1, coord2, unit=Unit.KILOMETERS)
    return total_distance

In [131]:
def calculate_trip_duration(trip_df):
    """Calculate duration of a trip in minutes."""
    start_time = trip_df['Timestamp(ms)'].iloc[0]
    end_time = trip_df['Timestamp(ms)'].iloc[-1]
    duration_sec = (end_time - start_time) / 1000
    return duration_sec / 60  # minutes

In [132]:
def plot_trip_map(trip_df, trip_number, save_dir=OUTPUT_MAPS_DIR):
    """Plot GPS trajectory of a trip using folium and save HTML map."""
    start_lat = trip_df['Latitude[deg]'].iloc[0]
    start_lon = trip_df['Longitude[deg]'].iloc[0]
    
    m = folium.Map(location=[start_lat, start_lon], zoom_start=13)
    
    coordinates = list(zip(trip_df['Latitude[deg]'], trip_df['Longitude[deg]']))
    
    folium.PolyLine(coordinates, color="blue", weight=3).add_to(m)
    folium.Marker(coordinates[0], popup="Start", icon=folium.Icon(color="green")).add_to(m)
    folium.Marker(coordinates[-1], popup="End", icon=folium.Icon(color="red")).add_to(m)
    
    map_path = Path(save_dir) / f"trip_{trip_number}_map.html"
    m.save(map_path)
    print(f"Map for trip {trip_number} saved as {map_path}")

In [133]:
trip_results = {}

for trip_number in data['Trip'].unique():
    trip_df = data[data['Trip'] == trip_number].reset_index(drop=True)
    
    distance = calculate_trip_distance(trip_df)
    duration = calculate_trip_duration(trip_df)
    
    trip_results[trip_number] = {
        "distance_km": distance,
        "duration_min": duration
    }
    
    plot_trip_map(trip_df, trip_number)

# Print summary
print("Trip Results:")
for trip_number, res in trip_results.items():
    print(f"Trip {trip_number}: {res['distance_km']:.2f} km | {res['duration_min']:.2f} min")

Map for trip 1558 saved as trip_maps\trip_1558_map.html
Map for trip 1561 saved as trip_maps\trip_1561_map.html
Map for trip 1567 saved as trip_maps\trip_1567_map.html
Map for trip 1568 saved as trip_maps\trip_1568_map.html
Map for trip 1572 saved as trip_maps\trip_1572_map.html
Map for trip 1578 saved as trip_maps\trip_1578_map.html
Map for trip 1582 saved as trip_maps\trip_1582_map.html
Map for trip 1585 saved as trip_maps\trip_1585_map.html
Trip Results:
Trip 1558: 0.82 km | 1.77 min
Trip 1561: 2.41 km | 7.67 min
Trip 1567: 1.47 km | 6.17 min
Trip 1568: 1.50 km | 5.23 min
Trip 1572: 1.97 km | 6.37 min
Trip 1578: 1.03 km | 3.89 min
Trip 1582: 0.68 km | 0.89 min
Trip 1585: 1.30 km | 3.99 min
